# 06 — Machine Learning: Delay Prediction (Classification)

**Airline Operations Intelligence Platform** · Notebook 6 of 10 · *runs locally*

## Purpose
Module 9 of the plan: predict whether a flight will arrive **more than 15 minutes late**,
using Spark MLlib. Logistic Regression and Random Forest, evaluated honestly against a
baseline.

## The two things that make or break this notebook

**1. Feature leakage.** The target is arrival delay. Any field only known *after* the
aircraft departs must be excluded — `dep_delay` alone predicts `arr_delay` almost
perfectly and would produce a meaningless 95% model. The plan lists the banned fields
explicitly and §2 below enforces that list in code.

**2. Historical rates must come from the training split only.** Features like "this
airport's historical delay rate" are computed *from the data*. Computing them over the
full dataset leaks test-set outcomes into training features — a subtle leak that inflates
scores and is easy to miss. §4 computes them on `train` alone and joins them to both
splits.

## Class imbalance
18.61% of completed flights are delayed. A model predicting "never delayed" scores
**81.39% accuracy** and is useless. Accuracy is therefore reported *alongside* its
baseline, and the decision metrics are precision, recall, F1 and ROC-AUC.

In [ ]:
import sys, time, json
sys.path.insert(0, "../src")

from config import build_spark, PATHS
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# Training holds several large cached DataFrames at once, so this notebook asks for
# more driver heap than the 3g default in src/config.py. Still well inside 8 GB.
spark = build_spark("06-classification", **{"spark.driver.memory": "5g"})

flights = spark.read.parquet(str(PATHS["curated"] / "flights.parquet"))

# Only completed flights have an arrival outcome. Cancelled/diverted flights are a
# different prediction problem and including them would corrupt the label.
data = flights.filter(F.col("status") == "completed")
print(f"Completed flights: {data.count():,}")

---
## 1. The target

`is_delayed` = 1 when arrival delay ≥ 15 minutes, the standard US DOT threshold,
already derived in notebook 02.

In [ ]:
label_dist = (data.groupBy("is_delayed").count()
                  .withColumn("pct", F.round(100.0*F.col("count")/data.count(), 2)))
label_dist.show()

pos_rate = data.agg(F.avg("is_delayed")).first()[0]
print(f"Positive class (delayed) : {100*pos_rate:.2f}%")
print(f"BASELINE accuracy of 'always predict on-time' : {100*(1-pos_rate):.2f}%")
print("\nAny model must beat that baseline on F1, not accuracy.")

---
## 2. Feature leakage control

The plan lists fields that must not be used. Rather than trusting a comment, the banned
list is declared in code and asserted against the feature set before training.

In [ ]:
# Known only AFTER departure -- using any of these leaks the answer.
BANNED = {
    "dep_delay", "is_delayed_dep",     # departure delay predicts arrival delay directly
    "actual_dep_min", "actual_arr_min",
    "taxi_out", "taxi_in",
    "air_time", "actual_duration",
    "arr_delay", "delay_category",     # the target itself, and a function of it
    "delay_carrier", "delay_weather", "delay_nas",
    "delay_security", "delay_late_aircraft",
    "cancellation_reason", "cancelled_after_pushback", "status",
}

# Known BEFORE departure -- legitimate predictors.
CATEGORICAL = ["airline_code", "time_of_day", "season"]
NUMERIC     = ["month", "day_of_week", "sched_dep_hour", "distance",
               "sched_duration", "is_weekend_int"]
HISTORICAL  = ["origin_delay_rate", "dest_delay_rate",
               "airline_delay_rate", "route_delay_rate"]

used = set(CATEGORICAL + NUMERIC + HISTORICAL)
assert not (used & BANNED), f"LEAKAGE: {used & BANNED}"
print("Leakage check passed. Features known before departure only.\n")
print(f"  categorical : {CATEGORICAL}")
print(f"  numeric     : {NUMERIC}")
print(f"  historical  : {HISTORICAL}  (computed from TRAIN only, see section 4)")

---
## 3. Stratified train/test split

Split **before** computing any feature derived from the data. Stratifying on the label
keeps the 18.61% positive rate identical in both splits.

**Implementation note.** The test set must be the *exact complement* of the training set.
`DataFrame.subtract()` cannot be used for this: it is a set difference over **distinct
rows**, so where thousands of flights share the same feature combination, subtracting the
training set removes all of them. That silently produces both the wrong split ratio and
the wrong class balance. A unique row id plus a `left_anti` join gives the true complement.

In [ ]:
base = data.select(
    "is_delayed", "airline_code", "origin", "destination", "route",
    "month", "day_of_week", "sched_dep_hour", "distance", "sched_duration",
    "time_of_day", "season",
    F.col("is_weekend").cast("int").alias("is_weekend_int"),
).na.drop().withColumn("row_id", F.monotonically_increasing_id())

# Spill to disk rather than fail when memory is tight -- the default MEMORY_ONLY
# silently drops partitions and forces recomputation, or OOMs the driver.
base = base.persist(StorageLevel.MEMORY_AND_DISK)
n_base = base.count()

SEED = 42

# Stratified: sample 80% within each label so the class ratio is preserved exactly.
train = base.sampleBy("is_delayed", {0: 0.8, 1: 0.8}, seed=SEED) \
            .persist(StorageLevel.MEMORY_AND_DISK)

# Exact complement by id -- NOT subtract(), see the note above.
test = base.join(train.select("row_id"), "row_id", "left_anti") \
           .persist(StorageLevel.MEMORY_AND_DISK)

n_train, n_test = train.count(), test.count()
r_train = train.agg(F.avg("is_delayed")).first()[0]
r_test  = test.agg(F.avg("is_delayed")).first()[0]

print(f"Train : {n_train:>9,}  positive rate {100*r_train:.2f}%")
print(f"Test  : {n_test:>9,}  positive rate {100*r_test:.2f}%")
print(f"Split : {100*n_train/(n_train+n_test):.1f}% / {100*n_test/(n_train+n_test):.1f}%")

assert n_train + n_test == n_base, "split is not a partition"
assert abs(r_train - r_test) < 0.005, "stratification failed: class rates differ"
print("\nSplit verified: exact partition, class balance preserved in both halves.")

---
## 4. Historical rate features — computed on TRAIN only

This is the step where leakage most often creeps in. "Historical delay rate for airport X"
is a *learned* quantity: if it is computed over the whole dataset, every training row
carries information about test-set outcomes and the model looks better than it is.

Smoothing is applied so that a route with 3 flights does not get an extreme rate — the
same small-sample concern as the rankings, expressed as a Bayesian prior toward the
global mean.

In [ ]:
GLOBAL_RATE = train.agg(F.avg("is_delayed")).first()[0]
SMOOTHING   = 100      # prior weight, in "virtual flights" at the global rate

def historical_rate(keys, out_col):
    """Smoothed delay rate per key group, learned from TRAIN only."""
    return (train.groupBy(*keys)
            .agg(F.count("*").alias("n"), F.avg("is_delayed").alias("rate"))
            .withColumn(out_col,
                (F.col("n") * F.col("rate") + F.lit(SMOOTHING * GLOBAL_RATE))
                / (F.col("n") + F.lit(SMOOTHING)))
            .select(*keys, out_col))

origin_rate  = historical_rate(["origin"], "origin_delay_rate")
dest_rate    = historical_rate(["destination"], "dest_delay_rate")
airline_rate = historical_rate(["airline_code"], "airline_delay_rate")
route_rate   = historical_rate(["route"], "route_delay_rate")

print(f"Global train delay rate : {GLOBAL_RATE:.4f}")
origin_rate.orderBy(F.desc("origin_delay_rate")).show(5)

In [ ]:
def add_history(df):
    return (df
        .join(F.broadcast(origin_rate),  "origin",       "left")
        .join(F.broadcast(dest_rate),    "destination",  "left")
        .join(F.broadcast(airline_rate), "airline_code", "left")
        .join(F.broadcast(route_rate),   "route",        "left")
        # Unseen keys in test fall back to the global rate rather than null.
        .fillna({"origin_delay_rate":  GLOBAL_RATE, "dest_delay_rate":   GLOBAL_RATE,
                 "airline_delay_rate": GLOBAL_RATE, "route_delay_rate":  GLOBAL_RATE}))

train_f = add_history(train).persist(StorageLevel.MEMORY_AND_DISK)
test_f  = add_history(test).persist(StorageLevel.MEMORY_AND_DISK)

print(f"Train with features : {train_f.count():,}")
print(f"Test  with features : {test_f.count():,}")

unseen = test_f.join(route_rate, "route", "left_anti").count()
print(f"Test rows whose route was unseen in train : {unseen:,} (fall back to global rate)")

---
## 5. Class weights

With 18.6% positives, an unweighted model minimises loss by predicting the majority class.
Weighting each class inversely to its frequency makes errors on the minority class as
costly as errors on the majority.

In [ ]:
w_pos = (1 - GLOBAL_RATE) / GLOBAL_RATE     # weight for the rare class
train_w = train_f.withColumn("weight", F.when(F.col("is_delayed") == 1, w_pos).otherwise(1.0))

print(f"Weight for delayed flights : {w_pos:.3f}")
print(f"Weight for on-time flights : 1.000")
train_w.groupBy("is_delayed").agg(F.count("*").alias("rows"),
                                  F.round(F.sum("weight"), 0).alias("total_weight")).show()

---
## 6. The MLlib pipeline

`StringIndexer` → `OneHotEncoder` → `VectorAssembler`. Only low-cardinality categoricals
are one-hot encoded; high-cardinality identities (322 airports, 4,706 routes) enter
through their historical rates instead, which keeps the feature vector small.

In [ ]:
indexers = [StringIndexer(inputCol=col, outputCol=f"{col}_idx", handleInvalid="keep")
            for col in CATEGORICAL]
encoders = [OneHotEncoder(inputCol=f"{col}_idx", outputCol=f"{col}_vec", handleInvalid="keep")
            for col in CATEGORICAL]

feature_cols = [f"{col}_vec" for col in CATEGORICAL] + NUMERIC + HISTORICAL
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")

prep = Pipeline(stages=indexers + encoders + [assembler]).fit(train_w)

train_v = prep.transform(train_w).select("features", "is_delayed", "weight") \
              .persist(StorageLevel.MEMORY_AND_DISK)
test_v  = prep.transform(test_f).select("features", "is_delayed") \
              .persist(StorageLevel.MEMORY_AND_DISK)

print(f"Feature vector size : {train_v.first()['features'].size}")
print(f"Train vectors : {train_v.count():,}   Test vectors : {test_v.count():,}")

# The vectors are materialised; the wide upstream frames are no longer needed.
for df in (base, train, test, train_f, test_f):
    df.unpersist()
print("Upstream frames released.")

---
## 7. Evaluation harness

Defined once so both models are judged identically, and so the confusion matrix and
baseline appear alongside every score.

In [ ]:
def evaluate(predictions, name, train_seconds):
    tp = predictions.filter("is_delayed = 1 AND prediction = 1").count()
    tn = predictions.filter("is_delayed = 0 AND prediction = 0").count()
    fp = predictions.filter("is_delayed = 0 AND prediction = 1").count()
    fn = predictions.filter("is_delayed = 1 AND prediction = 0").count()

    total     = tp + tn + fp + fn
    accuracy  = (tp + tn) / total
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0
    f1        = 2*precision*recall/(precision+recall) if (precision+recall) else 0.0
    auc = BinaryClassificationEvaluator(
        labelCol="is_delayed", rawPredictionCol="rawPrediction",
        metricName="areaUnderROC").evaluate(predictions)
    baseline = (tn + fp) / total     # always predict "on time"

    print(f"\n{'='*54}\n{name}\n{'='*54}")
    print(f"  Training time     : {train_seconds:.1f}s")
    print(f"  Accuracy          : {accuracy:.4f}   (baseline {baseline:.4f})")
    print(f"  Lift over baseline: {accuracy - baseline:+.4f}")
    print(f"  Precision         : {precision:.4f}")
    print(f"  Recall            : {recall:.4f}")
    print(f"  F1                : {f1:.4f}")
    print(f"  ROC-AUC           : {auc:.4f}")
    print(f"\n  Confusion matrix          predicted")
    print(f"                          on-time   delayed")
    print(f"    actual on-time      {tn:>9,} {fp:>9,}")
    print(f"    actual delayed      {fn:>9,} {tp:>9,}")

    return dict(model_name=name, accuracy=round(accuracy,4), precision=round(precision,4),
                recall=round(recall,4), f1=round(f1,4), roc_auc=round(auc,4),
                baseline_accuracy=round(baseline,4),
                true_positives=tp, true_negatives=tn,
                false_positives=fp, false_negatives=fn,
                training_seconds=round(train_seconds,1))

---
## 8. Logistic Regression

In [ ]:
t0 = time.time()
lr = LogisticRegression(labelCol="is_delayed", featuresCol="features",
                        weightCol="weight", maxIter=50, regParam=0.01)
lr_model = lr.fit(train_v)
t_lr = time.time() - t0

lr_metrics = evaluate(lr_model.transform(test_v), "Logistic Regression", t_lr)

---
## 9. Random Forest

Depth and tree count are bounded to fit the 8 GB budget. `maxBins` must be at least the
highest categorical cardinality after encoding.

In [ ]:
t0 = time.time()
rf = RandomForestClassifier(labelCol="is_delayed", featuresCol="features",
                            weightCol="weight", numTrees=40, maxDepth=10,
                            maxBins=64, seed=SEED, subsamplingRate=0.7)
rf_model = rf.fit(train_v)
t_rf = time.time() - t0

rf_metrics = evaluate(rf_model.transform(test_v), "Random Forest", t_rf)

---
## 10. Feature importance

Random Forest reports Gini importance, indexed by **position in the feature vector**.
Reconstructing those names by hand is error-prone: `OneHotEncoder(handleInvalid="keep")`
adds a category for unseen values and `dropLast` removes one, so a hand-built list
silently goes out of alignment and every importance is then attributed to the wrong
feature.

`VectorAssembler` writes the authoritative index→name mapping into the column's
`ml_attr` metadata. Read it from there.

In [ ]:
# Authoritative index -> name mapping, straight from the assembled vector's metadata.
attrs = train_v.schema["features"].metadata["ml_attr"]["attrs"]
index_to_name = {}
for kind in ("numeric", "binary", "nominal"):
    for a in attrs.get(kind, []):
        index_to_name[a["idx"]] = a["name"]

imp = rf_model.featureImportances.toArray()
assert len(index_to_name) == len(imp), \
    f"metadata has {len(index_to_name)} names but the model has {len(imp)} importances"

pairs = sorted(((index_to_name[i], v) for i, v in enumerate(imp)), key=lambda kv: -kv[1])

print(f"Feature vector width: {len(imp)}  (names resolved from metadata)\n")
print(f"{'FEATURE':<34}{'IMPORTANCE':>12}")
print("-" * 48)
for nm, v in pairs[:15]:
    print(f"{nm:<34}{v:>12.4f}  {'#' * int(v * 200)}")

feature_importances = {nm: round(float(v), 5) for nm, v in pairs[:20]}

---
## 11. Comparison and honest interpretation

In [ ]:
print(f"{'METRIC':<20}{'Logistic Reg':>15}{'Random Forest':>16}{'Baseline':>11}")
print("-"*62)
for key in ["accuracy","precision","recall","f1","roc_auc"]:
    b = f"{lr_metrics['baseline_accuracy']:.4f}" if key == "accuracy" else "-"
    print(f"{key:<20}{lr_metrics[key]:>15.4f}{rf_metrics[key]:>16.4f}{b:>11}")
print("-"*62)
print(f"{'training time':<20}{lr_metrics['training_seconds']:>14.1f}s"
      f"{rf_metrics['training_seconds']:>15.1f}s")

best = max([lr_metrics, rf_metrics], key=lambda m: m["f1"])
print(f"\nBest by F1: {best['model_name']} (F1 = {best['f1']:.4f})")

### Reading these numbers honestly

**Accuracy may sit near or below the 81.4% baseline. That is expected and is not a
failure.** Class weighting deliberately trades accuracy for recall: an unweighted model
would score ~81% by predicting "on time" almost always, catching nearly no delays. The
weighted model sacrifices some accuracy to actually identify delayed flights, which is
the useful behaviour for an operations dashboard.

**ROC-AUC is the fairest single summary here**, since it is insensitive to the decision
threshold and to class imbalance.

**Ceiling on performance.** The features available before departure — airline, airports,
schedule, distance and historical rates — cannot capture what actually causes delays on
the day: weather at departure time, the inbound aircraft running late (39.8% of delay
minutes, per notebook 05), air-traffic-control decisions, crew availability. A large
irreducible error is the correct result, not a modelling mistake. The weather enrichment
listed as an extension in the plan is the main avenue for improvement.

---
## 12. Persist model and results

In [ ]:
rf_model.write().overwrite().save(str(PATHS["models"] / "rf_delay_classifier"))
lr_model.write().overwrite().save(str(PATHS["models"] / "lr_delay_classifier"))
prep.write().overwrite().save(str(PATHS["models"] / "feature_pipeline"))
print("Models saved to", PATHS["models"])

In [ ]:
results = {
    "models": [lr_metrics, rf_metrics],
    "best_model": best["model_name"],
    "feature_importances": feature_importances,
    "train_rows": n_train, "test_rows": n_test,
    "positive_rate": round(GLOBAL_RATE, 4),
    "features_used": {"categorical": CATEGORICAL, "numeric": NUMERIC, "historical": HISTORICAL},
    "leakage_excluded": sorted(BANNED),
}

out = PATHS["marts"] / "ml_classification_results.json"
out.write_text(json.dumps(results, indent=2))
print("Wrote", out)

# Also as Parquet, so notebook 08 pushes it like any other mart.
spark.createDataFrame([lr_metrics, rf_metrics]).coalesce(1).write.mode("overwrite") \
     .parquet(str(PATHS["marts"] / "ml_classification_results.parquet"))
print("Wrote ml_classification_results.parquet")

In [ ]:
train_v.unpersist(); test_v.unpersist()
spark.stop()
print("Notebook 06 complete.")